# 05 — Live pipeline demo (the judging demo, section 12)

End-to-end flow, matching the architecture figure:

**telemetry stream → feature builder → Model 1 risk → decision gate → routing → Model 2 action → Granite explanation → dispatcher → audit log**

`analyse()` is the single function a future backend API endpoint would wrap; its output follows the API contract in section 11.

Before running: notebooks 01–03 must have produced the files in `artifacts/`.

### Granite (open source, runs locally)
Explanations come from an IBM Granite model (Apache-2.0) running in **llama.cpp** on this machine. No cloud API or API key
is involved, so no convoy data leaves LinuxONE. The model is found automatically: the `GRANITE_GGUF_PATH` env var,
then `models/`, then the Granite model preinstalled in the LinuxONE Jupyter container. If none is available, a
deterministic template is used and is **labelled as such**. Granite only *explains*; it never chooses the action or the route.

In [ ]:
RUN_ROUTING_DEMO = False
%run ./04_routing.ipynb

In [ ]:
import datetime as dt
import hashlib
import importlib.util
import os
import sys
import time
import joblib
import pandas as pd
from IPython.display import Markdown, display

spec = json.loads((ART / "feature_spec.json").read_text())
M1_FEATURES = spec["model1_features"]
M2_FEATURES = spec["model2_categorical"] + spec["model2_numeric"]
THRESHOLD = spec["risk_threshold"]
CALM = spec["calm_reference"]

model1 = joblib.load(ART / "model1_rf.joblib")
model2 = joblib.load(ART / "model2_rf.joblib")

LOG_DIR = ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
AUDIT_LOG = LOG_DIR / "audit_log.jsonl"

# --- Demo parameters ---
DISPATCHER_DECISION = "ACCEPT"     # the human in the loop: "ACCEPT" or "REJECT"
CARGO = {"description": "critical medical supplies", "priority": 5}
N_TICKS, INJECT_AT = 20, 6
rng = np.random.default_rng(7)

## Feature builder, Model 1 wrapper and per-prediction risk drivers

In [ ]:
def build_m1_features(telemetry, env, route_m):
    """Live, model-ready vector for one route. Weather is regional; hazard terms are route-specific."""
    on_hazard = route_m["hazard_exposure"] > 0.1
    return {
        "speed_kmh": telemetry["speed_kmh"],
        "fuel_level_pct": telemetry["fuel_level_pct"],
        "rainfall_mm_per_hr": env["rainfall_mm_per_hr"],
        "wind_speed_kmh": env["wind_speed_kmh"],
        "flood_risk": max(env["regional_flood_risk"], route_m["hazard_exposure"]),
        "traffic_congestion": env["traffic_congestion"],
        "distance_to_hazard_km": route_m["min_distance_to_hazard_km"],
        "route_delay_minutes": env["route_delay_minutes"] if on_hazard else 0.0,
        "road_closure_nearby": int(route_m["blocked"]),
    }


def predict_risk(features):
    return float(np.clip(model1.predict(pd.DataFrame([features])[M1_FEATURES])[0], 0, 1))


def risk_level(risk):
    return "HIGH" if risk >= THRESHOLD else "MEDIUM" if risk >= 0.4 else "LOW"


def risk_drivers(features, top_k=4):
    """How much the risk drops when each feature alone is reset to its calm value."""
    rows = [features] + [{**features, f: CALM[f]} for f in M1_FEATURES]
    preds = model1.predict(pd.DataFrame(rows)[M1_FEATURES])
    drops = {f: float(preds[0] - p) for f, p in zip(M1_FEATURES, preds[1:])}
    return {f: round(d, 3) for f, d in sorted(drops.items(), key=lambda kv: -kv[1])[:top_k] if d > 0.01}

## Explanation layer: Granite on llama.cpp, with a labelled fallback

GGUF model files store numbers in a fixed byte order. IBM Z is **big-endian**, so on LinuxONE you need a big-endian GGUF.
The Granite model preinstalled in the container already is one. A model downloaded from Hugging Face is little-endian
and must be converted with llama.cpp's `gguf_convert_endian.py <file> big`. The cell below checks this before loading.

In [ ]:
def _glob_granite(base, max_depth=4):
    for depth in range(max_depth + 1):
        yield from sorted(base.glob("*/" * depth + "*[Gg]ranite*.gguf"))


def find_granite_model():
    env_path = os.getenv("GRANITE_GGUF_PATH")
    if env_path:
        return Path(env_path) if Path(env_path).is_file() else None
    search = [ROOT / "models"]
    spec_ = importlib.util.find_spec("loan_demo")    # IBM's sample package in the LinuxONE container
    if spec_ and spec_.origin:
        search.append(Path(spec_.origin).parent)
    search += [Path("/home/ibm-user"), Path.home(), Path("/opt")]
    for base in search:
        if base.is_dir():
            match = next(_glob_granite(base), None)
            if match:
                return match
    return None


def gguf_byte_order(path):
    with open(path, "rb") as f:
        magic, version = f.read(4), f.read(4)
    if magic != b"GGUF":
        return "not a GGUF file"
    return "little" if int.from_bytes(version, "little") < 65536 else "big"


granite, GRANITE_PATH = None, find_granite_model()
if GRANITE_PATH is None:
    print("No Granite GGUF found; explanations will use the labelled template.")
else:
    order = gguf_byte_order(GRANITE_PATH)
    print(f"Granite model : {GRANITE_PATH} ({GRANITE_PATH.stat().st_size / 1e9:.2f} GB, {order}-endian)")
    if order != sys.byteorder:
        print(f"WARNING: this machine is {sys.byteorder}-endian; convert the model or use the container's Granite model.")
    else:
        try:
            from jinja2.sandbox import ImmutableSandboxedEnvironment
            from llama_cpp import Llama
            t0 = time.perf_counter()
            granite = Llama(model_path=str(GRANITE_PATH), n_ctx=2048, verbose=False)
            # Render the model's own chat template so reasoning ("thinking") mode can be switched off;
            # templates that have no such switch simply ignore the variable.
            GRANITE_TEMPLATE = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True).from_string(
                granite.metadata["tokenizer.chat_template"])
            print(f"Loaded with llama.cpp in {time.perf_counter() - t0:.1f} s using {granite.n_threads} threads")
        except Exception as exc:
            granite = None
            print(f"Could not load Granite ({exc}); explanations will use the labelled template.")

In [ ]:
FEATURE_LABELS = {
    "speed_kmh": "vehicle speed", "fuel_level_pct": "fuel level", "rainfall_mm_per_hr": "rainfall",
    "wind_speed_kmh": "wind", "flood_risk": "flood risk", "traffic_congestion": "congestion",
    "distance_to_hazard_km": "proximity to the hazard", "route_delay_minutes": "route delay",
    "road_closure_nearby": "a road closure on the route",
}


def template_explanation(facts):
    drivers = ", ".join(FEATURE_LABELS[f] for f in facts["risk_drivers"]) or "multiple factors"
    action = facts["recommended_action"]
    text = (f"Route risk is {facts['risk_score']:.2f} ({facts['risk_level']}), driven mainly by {drivers}. ")
    if action == "REROUTE":
        text += (f"An alternative route lowers risk to {facts['alternative_route_risk']:.2f} for "
                 f"+{facts['extra_detour_minutes']:.0f} min of travel, so rerouting is recommended for this "
                 f"cargo ({facts['cargo']}).")
    elif action == "DIVERT_TO_BACKUP_HUB":
        text += "No acceptable route to the destination was found, so diverting to the backup hub is recommended."
    elif action == "HOLD_POSITION":
        text += (f"The hazard is expected to last about {facts['estimated_hazard_duration_min']:.0f} minutes; "
                 "holding position is safer than the available alternatives.")
    else:
        text += "No intervention is needed; monitoring continues."
    return text


SYSTEM_PROMPT = (
    "You explain recommendations from a humanitarian logistics decision-support system to a human dispatcher. "
    "Write 2-3 plain sentences. Use ONLY the facts provided. Never recommend a different action, invent routes, "
    "or add numbers that are not in the facts."
)


def granite_explain(facts):
    """Returns (explanation, source, stats). Granite explains the decision; it never changes it."""
    if granite is None:
        return template_explanation(facts), "template (Granite not available)", {}
    prompt_facts = {**facts, "risk_drivers": [FEATURE_LABELS[f] for f in facts["risk_drivers"]]}
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Facts: {json.dumps(prompt_facts)}\nExplain the recommended action."}]
    try:
        prompt = GRANITE_TEMPLATE.render(
            messages=messages, add_generation_prompt=True, enable_thinking=False, bos_token="", eos_token="",
            strftime_now=lambda fmt: dt.datetime.now().strftime(fmt))
        t0 = time.perf_counter()
        out = granite.create_completion(prompt, temperature=0, max_tokens=160)
        seconds = time.perf_counter() - t0
        text = out["choices"][0]["text"].split("</think>")[-1].strip()
        stats = {"latency_s": round(seconds, 2),
                 "tokens_per_s": round(out["usage"]["completion_tokens"] / seconds, 1)}
        return text, f"{GRANITE_PATH.name} via llama.cpp (local)", stats
    except Exception as exc:
        return template_explanation(facts), f"template (Granite call failed: {exc})", {}

## `analyse()` — one convoy update in, one recommendation out

The "hold" option's risk assumes the hazard has passed after waiting; this is a prototype assumption and is labelled in the options table.

In [ ]:
def analyse(telemetry, env, current_path, destination, hazards, cargo, backup_hub):
    here = current_path[0]
    cur_m = route_metrics(G, current_path, hazards)
    cur_f = build_m1_features(telemetry, env, cur_m)
    risk = predict_risk(cur_f)
    result = {"truck_id": telemetry["truck_id"], "risk_score": round(risk, 3), "risk_level": risk_level(risk),
              "recommended_action": "CONTINUE_MONITORING", "features": cur_f}
    if risk < THRESHOLD:
        return result

    # Decision gate crossed: evaluate alternatives with the routing layer, then score them with Model 1
    alt_path = safest_route(G, here, destination)
    alt_ok = alt_path is not None and alt_path != current_path
    if alt_ok:
        alt_m = route_metrics(G, alt_path, hazards)
        alt_risk = predict_risk(build_m1_features(telemetry, env, alt_m))
        detour = max(0.0, alt_m["travel_min"] - cur_m["travel_min"])
    else:
        alt_risk, detour = 1.0, 0.0
    hub_path = safest_route(G, here, backup_hub)
    hub_risk = predict_risk(build_m1_features(telemetry, env, route_metrics(G, hub_path, hazards))) if hub_path else 1.0
    hold_risk = predict_risk({**CALM, "fuel_level_pct": telemetry["fuel_level_pct"]})

    m2_row = {
        "disruption_type": env["hazard_type"],
        "disruption_severity": max((hz["severity"] for hz in hazards), default=0.0),
        "current_route_risk": risk,
        "alternative_route_available": int(alt_ok),
        "alternative_route_risk": alt_risk,
        "extra_detour_minutes": detour,
        "fuel_level_pct": telemetry["fuel_level_pct"],
        "cargo_priority": cargo["priority"],
        "estimated_hazard_duration_min": env["hazard_duration_min"],
        "backup_hub_available": int(hub_path is not None),
    }
    X2 = pd.DataFrame([m2_row])[M2_FEATURES]
    action = str(model2.predict(X2)[0])
    proba = {str(c): round(float(p), 3) for c, p in zip(model2.classes_, model2.predict_proba(X2)[0])}

    options = [
        {"option": "Continue", "risk": risk, "extra_minutes": 0.0},
        {"option": "Hold (assumes hazard clears)", "risk": hold_risk, "extra_minutes": env["hazard_duration_min"]},
        {"option": "Reroute", "risk": alt_risk if alt_ok else None, "extra_minutes": detour if alt_ok else None},
        {"option": "Divert to hub", "risk": hub_risk if hub_path else None, "extra_minutes": None},
    ]
    facts = {
        "risk_score": round(risk, 2), "risk_level": result["risk_level"], "risk_drivers": risk_drivers(cur_f),
        "recommended_action": action, "alternative_route_risk": round(alt_risk, 2),
        "extra_detour_minutes": round(detour, 1), "cargo": cargo["description"],
        "estimated_hazard_duration_min": env["hazard_duration_min"],
    }
    explanation, source, llm_stats = granite_explain(facts)
    result.update(
        recommended_action=action, action_probabilities=proba, alternative_route_risk=round(alt_risk, 3),
        extra_detour_minutes=round(detour, 1), risk_drivers=facts["risk_drivers"], options=options,
        explanation=explanation, explanation_source=source, explanation_stats=llm_stats,
        alt_path=alt_path, hub_path=hub_path,
    )
    return result

## Tamper-evident audit log

Every recommendation and dispatcher decision is appended to `logs/audit_log.jsonl`. Each entry stores the SHA-256
hash of the previous entry, forming a hash chain. Editing or deleting any past entry breaks every hash after it,
so `verify_audit_log()` detects it. This supports the traceability requirement and the security story: decisions
are auditable, and all inference, including the LLM, runs locally on LinuxONE with no external API calls.

In [ ]:
GENESIS_HASH = "0" * 64


def _entry_hash(prev_hash, entry):
    return hashlib.sha256((prev_hash + json.dumps(entry, sort_keys=True, default=str)).encode()).hexdigest()


def audit(entry):
    lines = AUDIT_LOG.read_text().splitlines() if AUDIT_LOG.exists() else []
    prev = json.loads(lines[-1]).get("hash", GENESIS_HASH) if lines else GENESIS_HASH
    entry = {**entry, "prev_hash": prev}
    entry["hash"] = _entry_hash(prev, entry)
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(entry, sort_keys=True, default=str) + "\n")


def verify_audit_log(path=AUDIT_LOG):
    prev, count = GENESIS_HASH, 0
    for count, line in enumerate(path.read_text().splitlines(), 1):
        entry = json.loads(line)
        stored = entry.pop("hash", None)
        if entry.get("prev_hash") != prev or _entry_hash(prev, entry) != stored:
            return f"BROKEN at entry {count}"
        prev = stored
    return f"OK ({count} entries)"

## Run the scenario

A convoy travels normally → a flash flood is injected ahead of it → risk crosses the threshold →
alternatives are evaluated → Model 2 recommends → Granite explains → the dispatcher decides → everything is logged.

In [ ]:
origin, destination, hub = pick_endpoints(G)
baseline = fastest_route(G, origin, destination)
flood_node = baseline[int(len(baseline) * 0.6)]
FLOOD = make_hazard(G.nodes[flood_node]["lat"], G.nodes[flood_node]["lon"], radius_km=1.0, severity=0.9)
step = max(1, len(baseline) // (2 * N_TICKS))

apply_hazards(G, [])
path, hazards, fuel, holding = list(baseline), [], 68.0, False
history, decision = [], None

for tick in range(N_TICKS):
    if tick == INJECT_AT:
        hazards = [FLOOD]
        apply_hazards(G, hazards)
    flood = bool(hazards)
    telemetry = {"truck_id": "RZ-01", "speed_kmh": 0.0 if holding else float(rng.normal(28 if flood else 62, 3)),
                 "fuel_level_pct": fuel}
    env = {"rainfall_mm_per_hr": 30.0 if flood else 1.5, "wind_speed_kmh": 40.0 if flood else 12.0,
           "regional_flood_risk": 0.15 if flood else 0.03, "traffic_congestion": 0.45 if flood else 0.2,
           "route_delay_minutes": 35.0 if flood else 2.0, "hazard_type": "FLOOD", "hazard_duration_min": 240}

    res = analyse(telemetry, env, path, destination, hazards, CARGO, hub)
    history.append({"tick": tick, "risk": res["risk_score"], "action": res["recommended_action"]})

    if decision is None and res["recommended_action"] != "CONTINUE_MONITORING":
        decision = {**res, "tick": tick, "original_path": list(path)}
        audit({"timestamp": dt.datetime.now(dt.timezone.utc).isoformat(), "tick": tick,
               **{k: v for k, v in res.items() if k not in ("alt_path", "hub_path")},
               "dispatcher_decision": DISPATCHER_DECISION})
        if DISPATCHER_DECISION == "ACCEPT":
            if res["recommended_action"] == "REROUTE":
                path = res["alt_path"]
            elif res["recommended_action"] == "DIVERT_TO_BACKUP_HUB":
                path, destination = res["hub_path"], hub
            elif res["recommended_action"] == "HOLD_POSITION":
                holding = True

    if not holding:
        path = path[step:] if len(path) > step else path[-1:]
        fuel -= 0.4

pd.DataFrame(history).set_index("tick").T

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(h["tick"], h["risk"], marker="o")
ax.axhline(THRESHOLD, color="red", ls="--", label=f"intervention threshold ({THRESHOLD})")
ax.axvline(INJECT_AT, color="tab:blue", ls=":", label="flood injected")
if decision:
    ax.axvline(decision["tick"], color="tab:green", ls=":",
               label=f"{decision['recommended_action']} ({DISPATCHER_DECISION.lower()}ed)")
ax.set_ylim(0, 1)
ax.set_xlabel("tick")
ax.set_ylabel("route risk (Model 1)")
ax.legend(loc="upper left")
ax.set_title("Live route risk for convoy RZ-01")
plt.tight_layout()

## What the dispatcher sees

In [ ]:
if decision is None:
    print("Risk never crossed the threshold; no recommendation was made.")
else:
    display(Markdown(f"**Tick {decision['tick']}: risk {decision['risk_score']} ({decision['risk_level']}) "
                     f"→ recommended action: `{decision['recommended_action']}`**"))
    display(pd.DataFrame(decision["options"]).set_index("option").round(2))
    print("Model 2 class probabilities:", decision["action_probabilities"])
    print("Top risk drivers (risk drop if reset to calm):", decision["risk_drivers"])
    display(Markdown(f"> {decision['explanation']}\n\n*Explanation source: {decision['explanation_source']}*"))
    if decision["explanation_stats"]:
        print("Granite latency: {latency_s} s, {tokens_per_s} tokens/s".format(**decision["explanation_stats"]))
    print("Dispatcher decision:", DISPATCHER_DECISION)

    alt_risk = decision["alternative_route_risk"]
    display(pd.DataFrame({
        "metric": ["Current route risk", "Alternative route risk", "Relative risk reduction",
                   "Additional travel time", "Cargo"],
        "value (simulation output)": [decision["risk_score"], alt_risk,
                                      f"{100 * (1 - alt_risk / decision['risk_score']):.1f}%",
                                      f"+{decision['extra_detour_minutes']:.0f} min", CARGO["description"]],
    }).set_index("metric"))

In [ ]:
if decision:
    plot_network(G, {
        "original route": (decision["original_path"], "tab:red"),
        "recommended alternative": (decision.get("alt_path"), "tab:green"),
        "backup hub route": (decision.get("hub_path"), "tab:orange"),
    }, hazards, title="Route options at decision time")

## API contract preview (section 11)

This is the response a `POST /analyse` endpoint would return once the backend is containerised on LinuxONE.

In [ ]:
API_FIELDS = ["truck_id", "risk_score", "risk_level", "recommended_action", "alternative_route_risk",
              "extra_detour_minutes", "explanation", "explanation_source"]
if decision:
    print(json.dumps({k: decision.get(k) for k in API_FIELDS}, indent=2))

## Verify the audit log, then show that tampering is detected

In [ ]:
if AUDIT_LOG.exists():
    print("Audit log:", AUDIT_LOG)
    print("Original log :", verify_audit_log())

    # Simulate someone quietly rewriting the most recent recommendation
    lines = AUDIT_LOG.read_text().splitlines()
    forged = json.loads(lines[-1])
    forged["recommended_action"] = "CONTINUE_MONITORING"
    tampered = LOG_DIR / "audit_log_tampered.jsonl"
    tampered.write_text("\n".join(lines[:-1] + [json.dumps(forged, sort_keys=True)]) + "\n")
    print("Tampered copy:", verify_audit_log(tampered))
    tampered.unlink()